Instalaçao da biblioteca necessaria (EXECUÇÃO OBRIGATORIA)



In [39]:
#!pip install deap pandapower rich streamlit

---
## 1) Parâmetros do JSON: (12/05/2025)
---

O arquivo JSON fornece os parametros necessarios para o algoritmo evolutivo.

Abaixo, a descrição de cada parâmetro:

1. **ARRAY_VAR**: Array das variaveis de decisão do problema podendo ser float ou int

2. **LIMITE_VAR**: Valores minimo e maximo das variaveis de decisao em um array

2.  **`CROSSOVER`:** Probabilidade de cruzamento (0 a 1 em porcentagem). Determina a chance de dois indivíduos pais trocarem material genético para produzir descendentes. Valores mais altos promovem a exploração, criando descendentes mais diversos.

3.  **`MUTACAO`:** Probabilidade de mutação (0 a 1 em porcentagem). Determina a chance de os genes de um indivíduo serem alterados aleatoriamente, introduzindo novas variações na população e prevenindo convergência prematura.

4.  **`NUM_GENERATIONS`:** Número total de gerações que o algoritmo evolutivo executará.

5.  **`POP_SIZE`:** Tamanho da população, ou seja, o número de indivíduos em cada geração. Populações maiores geralmente oferecem maior diversidade, mas exigem mais recursos computacionais.

6.  **`IND_SIZE`:** Tamanho do cromossomo de cada indivíduo.

7.  **`RCE_REPOPULATION_GENERATIONS`:** Intervalo (em gerações) em que a estratégia RCE (Repopulation with Elite Set) é aplicada. O RCE ajuda a manter a diversidade e evitar a estagnação, introduzindo novos indivíduos com base em critérios específicos.

8.  **`NUM_VAR_DIFERENTES`:** Define um limite para a diferença entre as variáveis de decisão dos indivíduos para serem considerados para inclusão no conjunto de elite durante o RCE.

9.  **`PORCENTAGEM`:** Determina a proporção da população considerada para o conjunto de elite durante o RCE. Ajuda a controlar a pressão de seleção e o equilíbrio entre exploração e aproveitamento.

10. **`DELTA_MIN`:** Valor de tolerância ou limite entre valores das mesmas variáveis de decisão em indivíduos comparados pela RCE.

In [40]:
import json
def load_params(file_path):
    with open(file_path, "r") as file:
        params = json.load(file)
    return params


array_decisions =  [14,15,14,18,15]

#! 12/05/2025 - PVRV
# Testar co crossover em 90% e Mutação em 15% POP_SIZE = 50, IND_SIZE = 5, RCE_GEN = 5

params = {
    "ARRAY_VAR": array_decisions,
    'LIMITE_VAR': [0, 31],

    'NUM_GENERATIONS': 100,#300
    'CROSSOVER': 0.90,
    'MUTACAO': 0.5,

    'POP_SIZE': 100,#100
    'IND_SIZE': 5,

    'RCE_REPOPULATION_GENERATIONS': 25,

    'NUM_VAR_DIFERENTES': 1,
    'PORCENTAGEM': 0.2,
    'DELTA_MIN': 2
  }


### Testes do framework (31/03/25)
- importante analisar o comportamento do RCE
- analisar os valores global da função objetivo
- aumento de mutação e de gerações me busca de XMEN
- os 3 criterios do RCE tem que estar mais explicações

In [41]:
configuracoes_execucoes = {
        "key": True,
        "value": 5,
        "parametros_opcionais": [
             {"MUTACAO": [90,80,70]},
             {"CROSSOVER": [90,80,70]},
             {'NUM_GENERATIONS': [100, 200, 300]},

        ]
}


----
## 2) Setup
----


In [42]:
import numpy as np
import math
from deap import base, creator, tools
import random
import matplotlib.pyplot as plt
import time
from scipy.optimize import minimize
import json
import pandas as pd
import array


#! WARN (12/05/2025) - Usado fora da classes para NAO ter logs no output

# Criando os individuos e uma função e minimização
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))

creator.create("Individual", list, fitness=creator.FitnessMin,rce=str, index=int)

class Setup:
    def __init__(self, params, fitness_function, tamanho_hash = 0 ):

        #! Parametros JSON
        self.params = params
        self.CXPB = params["CROSSOVER"]
        self.MUTPB = params["MUTACAO"]
        self.NGEN = params["NUM_GENERATIONS"]

        # População de individuos com RCE
        self.POP_SIZE = params["POP_SIZE"]
        self.SIZE_INDIVIDUAL = params["IND_SIZE"]
        self.TAXA_GENERATION = params["RCE_REPOPULATION_GENERATIONS"]

        # Variaveis AG e AE
        self.CROSSOVER, self.MUTACAO, self.NUM_GENERATIONS, self.POPULATION_SIZE = (
            self.CXPB,
            self.MUTPB,
            self.NGEN,
            self.POP_SIZE,
        )

        #! Daodos de etrada do usuario nova
        self.limite = params["LIMITE_VAR"]
        self.decision_variables = params["ARRAY_VAR"]

        # Criterios Rainer DEAP
        self.NUM_VAR_DIF = params["NUM_VAR_DIFERENTES"]
        self.porcentagem = params["PORCENTAGEM"]
        self.delta = params["DELTA_MIN"]

        #!Criando individuo pelo deap com seus atributos
        self.toolbox = base.Toolbox()

        #! Parâmetros do algoritmo de Rastrigin
        self.evaluations = 0
        self.num_repopulation = int(self.NUM_GENERATIONS * (self.TAXA_GENERATION/100))

        # dict para acumular
        self.dataset = {}


        #creator.create("Individual", list, fitness=creator.FitnessMin, rce=str, index=int                 )

        #----------------------------------------------------------------------------------------
        # Correção 03/04/25 - Usando as variaveis de decisao no JSON

        def checkBounds(min, max):
            def decorator(func):
                def wrapper(*args, **kargs):
                    offspring = func(*args, **kargs)
                    for child in offspring:
                        for i in range(len(child)):
                            if child[i] > max:
                                child[i] = max
                            elif child[i] < min:
                                child[i] = min

                            if type(self.decision_variables[i]) is int:
                                child[i] = int(child[i])

                            elif type(self.decision_variables[i]) is float:
                                child[i] = float(child[i])

                    return offspring
                return wrapper
            return decorator


        # No frontend em Streamlit eu quero um editor online do JSON
        if self.decision_variables is not None:
            for i in range(len(self.decision_variables)):

                if type(self.decision_variables[i]) is int:
                    #print("Verificando valores inteiros na variaveis de decisão")

                    creator.create("Individual", list, fitness=creator.FitnessMin,rce=str, index=int)

                    self.toolbox.register(
                        "attribute", random.randint, self.limite[0], self.limite[1]
                    )

                    #! Update 25/04
                    # Mutação para variáveis inteiras
                    self.toolbox.register("mutate", tools.mutUniformInt, low=self.limite[0], up=self.limite[1], indpb=1/len(self.decision_variables))
                    #self.toolbox.register("mutate", tools.mutShuffleIndexes, indpb=1/len(self.decision_variables))

                elif type(self.decision_variables[i]) is float:
                    #print("Verificando valores float na variaveis de decisão")

                    creator.create("Individual",list,fitness=creator.FitnessMin,rce=str, index=int)

                    self.toolbox.register(
                        "attribute", random.uniform, int(self.limite[0]), int(self.limite[1])
                    )

                    # Mutação para variáveis float (mantém mutGaussian)
                    self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.1)


        else:
            raise ValueError("No arquivo JSON as variáveis de decisão não pode ser vazia.")

        # Define os tipos de variáveis de decisão (METOOD ANTIGO)
        #self.toolbox.register("attr_float", random.uniform, self.limite[0], self.limite[1])  # x: float entre -5.12 e 5.12
        #self.toolbox.register("attr_float", random.uniform, 0.0, 31.0)
        #self.toolbox.register("attr_int", random.randint, int(self.limite[0]), int(self.limite[1]))      # entre 0 ate 31 na funcao objeitvo

        #! registrando os individuos
        #self.toolbox.register("individual", creator.Individual, self.decision_variables, self.toolbox.attribute)
        self.toolbox.register("individual", tools.initRepeat, creator.Individual, self.toolbox.attribute, n=self.SIZE_INDIVIDUAL)

        #! criando e regsitrando a população de individuos (ja no type do deap)
        self.toolbox.register(
            "population", tools.initRepeat, creator.Individual, self.toolbox.individual
        )
        self.POPULATION = self.toolbox.population(n=self.POP_SIZE)

        #! paramentos evolutivos registrados
        self.toolbox.register("mate", tools.cxTwoPoint)
        # self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.1)
        self.toolbox.register("select", tools.selTournament, tournsize=3)

        # decorator
        self.toolbox.decorate("mate", checkBounds(self.limite[0], self.limite[1]))
        self.toolbox.decorate("mutate", checkBounds(self.limite[0], self.limite[1]))



        # Store the original fitness function
        self.funcao_objetivo = fitness_function

        # Use the original function if provided, otherwise use rastrigin
        self.__fitness_function = fitness_function if fitness_function is not None else self.rastrigin

        #! Register the fitness function using a lambda function, directly referencing the stored function
        self.toolbox.register("evaluate", self.funcao_objetivo if self.funcao_objetivo else self.rastrigin)
        #self.toolbox.register("evaluate", fitness_func)

        # Teste para validar dados de entrada
        self.checkDecisionVariablesAndFitnessFunction(
            self.__fitness_function,
            self.POPULATION[0],
        )


        if tamanho_hash > 0:
            self.tabela_hash = [-1] * tamanho_hash
        else:
            self.tabela_hash = None

    #     # Aqui já pode carregar sistema teste e dados desligamentos
    #     self.carregar_dados()

    # def carregar_dados(self):
    #     # Exemplo: carregar sistema IEEE14, desligamentos, etc
    #     pass

    def avaliarFitnessIndividuos(self, pop):
        fitnesses = []  # To store fitness values for each individual
        for ind in pop:
            fitness = self.toolbox.evaluate(list(ind), self) # Assuming this calls funcao_objetivo_IEEE14
            ind.fitness.values = (fitness,)
            fitnesses.append(fitness)  # Add fitness value to the list
        return fitnesses
        #print("\nFitness individuos validados!!! ")


    def checkDecisionVariablesAndFitnessFunction(
        self, fitness_function, individual
    ):
            self.__fitness_function = fitness_function

            # Criando o esqueleto de uma funcao objetivo com uma variavel de decisao
            def fitness_func(individual,self):
                return (
                    self.funcao_objetivo(individual,self)
                    if self.funcao_objetivo
                    else self.rastrigin(individual)
                )

            #! Registrar a função de fitness no toolbox
            print("\n[DEBUG] Dados do problema = ", self.decision_variables, self.__fitness_function)
            self.toolbox.register("evaluate", fitness_func)



    def gerarDataset(self, excel):
        df = pd.read_excel(excel)
        print(df.columns)
        self.dataset = {
            "CXPB": self.CROSSOVER,
            "TAXA_MUTACAO": self.MUTACAO,
            "NUM_GEN": self.NUM_GENERATIONS,
            "POP_SIZE": self.POPULATION_SIZE,
            "IND_SIZE": self.SIZE_INDIVIDUAL,
            "evaluations": self.evaluations,
            "NUM_REPOPULATION": self.num_repopulation,
        }

    def rastrigin(self, individual):
        self.evaluations += 1
        rastrigin = 10 * self.SIZE_INDIVIDUAL

        for i in range(self.SIZE_INDIVIDUAL):
            rastrigin += individual[i] * individual[i] - 10 * (
                math.cos(2 * np.pi * individual[i])
            )
        return rastrigin

    def rastrigin_decisionVariables(self, individual ):
        self.evaluations += 1
        rastrigin = 10 * len(individual)

        for i in range(len(individual)):
            rastrigin += individual[i] * individual[i] - 10 * (
                math.cos(2 * np.pi * individual[i])
            )
        return rastrigin

    def rosenbrock(self, x):

        var = np.array(x)

        return np.sum(100 * (var[1:] - var[:-1] ** 2) ** 2 + (1 - var[:-1]) ** 2)



/home/pedrov12/.local/lib/python3.13/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/pedrov12/.local/lib/python3.13/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


---
## 3) Algoritmo Evolutivo RCE
---






### Dashboard

In [43]:

# gera o arquivo para rodar dentro do colab

##%%writefile app.py

import pathlib
import streamlit as st
import pandas as pd
import plotly.graph_objects as go

import pickle
import plotly.io as pio
import os

import numpy as np
import matplotlib.pyplot as plt

#!pip install streamlit pandas plotly openpyxl

#from .Setup import params
import json

def load_params(file_path):
    """Carrega os parâmetros de um arquivo JSON."""
    with open(file_path, 'r') as file:
        params = json.load(file)
    return params


#params = load_params(r"C:\Users\Pedro Victor R V\Documents\GitHub\Repopulation-With-Elite-Set\src\AlgEvolutivoRCE\params.json")
#BASE_DIR = pathlib.Path(__file__).resolve().parent 
#print(BASE_DIR)
#params = load_params(f"{BASE_DIR}/params.json")



def get_folder_path():
    BASE_DIR = pathlib.Path(__file__).resolve().parent.parent.parent  

    # Define o caminho relativo para a pasta "output" dentro do projeto
    FOLDER_NAME = BASE_DIR / "src" / "output"

    # Cria a pasta "output" se ela não existir
    FOLDER_NAME.mkdir(parents=True, exist_ok=True)
    print("\nFOLDER_NAME [DEBUG] =", FOLDER_NAME)
    #print("FOLDER RAIZ =", FOLDER_NAME.parent)

    return FOLDER_NAME

#FOLDER_NAME = get_folder_path()

class DashboardApp:
    """Classe principal para criar o dashboard interativo com Streamlit."""

    def __init__(self,options):
        st.set_page_config(layout="wide", page_title="Dashboard Interativo")
        self.df = None
        self.fit_array = []
        self.optirons = options

    # codigo antigo

    def generateSimpleDataset(self):
        # Geração dos dados
        data = pd.DataFrame(
            {"x": np.linspace(-5, 5, 400), "y": np.linspace(-5, 5, 400)}
        )
        # Generate meshgrid data
        x = np.linspace(-5.15, 5.15, 100)
        y = np.linspace(-5.15, 5.15, 100)
        X, Y = np.meshgrid(x, y)

        # Calculate function values
        # print(X.shape,Y.shape)
        return X, Y


    def default_rastrigin(self, x, y):
        return 20 + x**2 + y**2 - 10 * (np.cos(2 * np.pi * x) + np.cos(2 * np.pi * y))

    def plot_Rastrigin_2D(self, X, Y, Z_rastrigin, logbook, best_variables=[]):
        fig = plt.figure(figsize=(18, 10))
        ax1 = fig.add_subplot(231)
        generation = logbook.select("gen")
        statics = self.calculate_stats(logbook)
        title = f"Estrategia RCE - Crossover: {params['CROSSOVER']*100}% e Mutação: {params['MUTACAO']*100}% " if len(generation) > 1 else "Sem Repopulação RCE"

        line1 = ax1.plot(
            generation, statics["min_fitness"], "*b-", label="Minimum Fitness"
        )
        line2 = ax1.plot(
            generation, statics["avg_fitness"], "+r-", label="Average Fitness"
        )
        line3 = ax1.plot(
            generation, statics["max_fitness"], "og-", label="Maximum Fitness"
        )
        ax1.set_xlabel("Generations")
        ax1.set_ylabel("Func. Fitness")
        ax1.set_title(title)
        lns = line1 + line2 + line3
        labs = [l.get_label() for l in lns]
        ax1.legend(lns, labs, loc="upper right")

        #! Graficos barras
        ax3 = fig.add_subplot(232)

        if len(generation) > 1:
            best_solutions = [
                min(statics["min_fitness"]) for i in range(len(generation))
            ]
            avg_fitness = statics["avg_fitness"]
            generations = np.arange(1, len(generation) + 1)

            ax3.plot(
                generations,
                avg_fitness,
                marker="o",
                color="r",
                linestyle="--",
                label="Média Fitness por Geração",
            )
            ax3.bar(
                generations,
                statics["min_fitness"],
                color="green",
                label="Melhor Fitness por Geração",
            )
            ax3.set_title("Best Fitness por Geração")
            ax3.set_xlabel("Geração")
            ax3.set_ylabel("Fitness")
            ax3.legend()

        #! Rastrigin 3D (rainer nao gosta)
        #ax5 = fig.add_subplot(233, projection="3d")
        #ax5.plot_surface(X, Y, Z_rastrigin, cmap="viridis", edgecolor="none")
        #ax5.set_title("Rastrigin Function 3D")
        #ax5.set_xlabel("X")
        #ax5.set_ylabel("Y")
        #ax5.set_zlabel("Z")

        plt.tight_layout()
        plt.show()

    def show_rastrigin_benchmark(self, logbook, best=[]):
        X, Y = self.generateSimpleDataset()

        Z_3D_rastrigin = self.default_rastrigin(X, Y)

        self.plot_Rastrigin_2D(X, Y, Z_3D_rastrigin, logbook, best)

    def graficoRCE(self, gen, lista,  repopulation=False):
        title = f"Estrategia RCE - Crossover: {params['CROSSOVER']*100}% e Mutação: {params['MUTACAO']*100}% " if repopulation else "Sem Repopulação RCE"

        # Create a new figure before plotting
        figure = go.Figure()   # Create new figure if None

        figure.add_trace(
            go.Scatter(
                x=gen,
                #y=lista["min_fitness"],
                y = lista[0],
                mode="lines+markers",
                name="Valor Min Fitness",
                marker=dict(symbol="star", color="blue"),
                line=dict(color="blue"),
            )
        )
        

        figure.add_trace(
            go.Scatter(
                x=gen,
                #y=lista["avg_fitness"],
                y = lista[1],

                mode="lines+markers",
                name="Média Fitness",
                marker=dict(symbol="cross", color="red"),
                line=dict(color="red"),
            )
        )
        
        
        figure.add_trace(
            go.Scatter(
                x=gen,
                #y=lista["max_fitness"],
                y = lista[2],

                mode="lines+markers",
                name="Valor Max Fitness",
                marker=dict(symbol="circle", color="green"),
                line=dict(color="green"),
            )
        )
       

        figure.update_traces(overwrite= True)

        figure.update_layout(
            title=title,
            xaxis_title="Generation",
            yaxis_title="Fitness",
            legend_title="Algoritimo Evolutivo",
            template="seaborn", #      ['ggplot2', 'seaborn', 'simple_white', 'plotly', 'plotly_white', ...]
            overwrite= True
            )

        return figure

    # --- visualize MODIFICADO (salva dados e chama Streamlit) ---
    def visualize(self, logbook, pop, repopulation=True, DEBUG=False, current_params=None, execution_num=1):
        """
        Processa dados, imprime no console, RETORNA figura e resultados,
        E salva os arquivos de dados e figura com um número de execução.
        """
        generation = []
        statics = {}
        array_values = []

        best_solution_index = -1
        best_solution_variables = []
        best_solution_fitness = float('inf')

        try:
            generation = logbook.select("gen")
            statics = self.calculate_stats(logbook)

            if DEBUG:
                print("\n\nDEBUG: Dados para gráfico (generation x statics)")
                print(len(statics["min_fitness"])) # DEBGU = 90

            min_fitness_values = statics.get("min_fitness", [])
            #print(len(min_fitness_values))
            if not min_fitness_values: raise ValueError("Min fitness list is empty")
            best_solution_fitness = min(min_fitness_values)
            best_solution_index = min_fitness_values.index(best_solution_fitness)

            
            # Foi feito os SLICING  porque o deap acumula os resultados  de fitness na varaiveal statics
            array_values.append(
                statics.get("min_fitness", [])[(execution_num -1) * int(len(generation)/execution_num):(execution_num * int(len(generation)/execution_num)) -1] 
            )
            array_values.append(
                statics.get("avg_fitness", [])[(execution_num -1) * int(len(generation)/execution_num):(execution_num * int(len(generation)/execution_num)) -1] 
            )
            array_values.append(
                statics.get("max_fitness", [])[(execution_num -1) * int(len(generation)/execution_num):(execution_num * int(len(generation)/execution_num)) -1] 
            )


            if repopulation:
                best_solution_variables = pop[0] if pop else []
            else:
                print("WARN: Visualize - Lógica para 'best_solution_variables' sem repopulação usa fallback.")
                best_solution_variables = pop[0] if pop else []

            print("\n\n")
            print("="*90)
            print(f"  >>> Soluções do problema (Execução {execution_num} - Console Output) <<<")
            print("="*90)
            print(f"Best Generation: {best_solution_index}")
            print(f"Best Variables: {best_solution_variables}")
            print(f"Best Fitness: {best_solution_fitness}")
            print("="*90)

            grafico_RCE = self.graficoRCE(generation, array_values, repopulation=repopulation)


             # --- MODIFIED: Define filenames based on execution_num ---
            if execution_num is None:

                print("WARN: execution_num not provided. Using default filenames.")

                raise ValueError("Execution number (execution_num) must be provided to visualize for saving files.")

            else:

                # check se o diretorio output exists
                output_path = f"{FOLDER_NAME}"

                data_file = f"{output_path}/dashboard_data_{execution_num}.pkl"

                fig_file = f"{output_path}/dashboard_fig_{execution_num}.json"

                # --- Salvar dados e figura para o script Streamlit ---
                data_to_save = {
                    'execution_num': execution_num, # Store execution number in data
                    'best_gen_idx': best_solution_index,
                    'best_vars': best_solution_variables,
                    'best_fitness': best_solution_fitness,
                    'params': current_params if current_params else params,
                    'logbook_data': {
                        'generation': generation,
                        'statics': statics
                    }
                }

            # Salvar a figura em formato HTML
            fig_filename = f"{output_path}/grafico_execucao_{execution_num}.html"
            grafico_RCE.write_html(fig_filename)


            # Salva os dados no arquivo .pkl numerado
            with open(data_file, 'wb') as f:
                pickle.dump(data_to_save, f)


            # Salva a figura no arquivo .json numerado
            fig_json = pio.to_json(grafico_RCE)
            with open(fig_file, 'w') as f:
                f.write(fig_json)

                
            print(f"\n[INFO]: Dados e figura para execução {execution_num} salvos com sucesso.") # Added execution num here
            print(fig_filename)

        except Exception as e:
            print(f"\n\n\nERRO em visualize (Execução {execution_num}): {e}") # Added execution num here
            st.sidebar.info("Erro em visualize: ",e)
            return -1, [], float('inf'), None

        return best_solution_index, best_solution_variables, best_solution_fitness, grafico_RCE

    def calculate_stats(self, logbook):

        fit_avg = logbook.select("avg")
        fit_std = logbook.select("std")
        fit_min = logbook.select("min")
        fit_max = logbook.select("max")

        self.fit_array.append(fit_min)
        self.fit_array.append(fit_avg)
        self.fit_array.append(fit_max)
        self.fit_array.append(fit_std)

        return {
            "min_fitness": fit_min,
            "max_fitness": fit_max,
            "avg_fitness": fit_avg,
            "std_fitness": fit_std,
        }

    def statistics_per_generation_df(self, logbook, save = True):
        generations = logbook.select("gen")
        min_fitness = logbook.select("min")
        avg_fitness = logbook.select("avg")
        max_fitness = logbook.select("max")
        std_fitness = logbook.select("std")

        data = {
            "Generation": generations,
            "Min Fitness": min_fitness,
            "Average Fitness": avg_fitness,
            "Max Fitness": max_fitness,
            "Std Fitness": std_fitness,
        }

        df = pd.DataFrame(data)
        if save:
            df.to_excel(f"{FOLDER_NAME}/statistics_RCE.xlsx", index=False)

        return avg_fitness, std_fitness


In [44]:
import numpy as np
import math
from deap import base, creator, tools
import random
import matplotlib.pyplot as plt
import time
import json
import pandas as pd
from scipy.optimize import minimize


class AlgoritimoEvolutivoRCE:

    def __init__(self, setup, DEBUG = True):
        self.setup = setup
        self.dashboard = DashboardApp(
            options= configuracoes_execucoes
        )
        self.DEBUG = DEBUG
        
        self.stats = tools.Statistics(key=lambda ind: ind.fitness.values)
        self.stats.register("avg", np.mean)
        self.stats.register("std", np.std)
        self.stats.register("min", np.min)
        self.stats.register("max", np.max)

        self.logbook = tools.Logbook()
        self.hof = tools.HallOfFame(1)
        self.POPULATION = self.setup.toolbox.population(n=self.setup.POP_SIZE)
        self.hof.update(self.POPULATION)

        self.pop_RCE = []
        self.best_solutions_array = []
        self.best_individual_array = []
        self.allIndividualValuesArray = []
        self.data = {}
        self.repopulation_counter = 0
        self.allFitnessValues = {}
        self.validateCounter = 0
        self.CONJUNTO_ELITE_RCE = set()

        self.decision_variables = []
        self.fitness_function  = lambda x: 0

    def registrarDados(self, generation):

        # Registrar estatísticas e melhores soluções
        for ind in self.POPULATION:
            avg_fitness_per_generation = np.mean(ind.fitness.values)
            std_deviation = np.std(ind.fitness.values)

        #! PEgandos os dados e colocando no df
        self.data = {
            "Generations": generation + 2,
            "Variaveis de Decisão": self.hof[0],
            "Evaluations": self.setup.evaluations,
            "Ind Valido": self.hof[0].fitness.valid,
            "Best Fitness": self.hof[0].fitness.values,
            "Media": avg_fitness_per_generation,
            "Desvio Padrao": std_deviation,
        }

        self.best_individual_array.append(self.data)

        self.visualizarPopAtual(generation, [avg_fitness_per_generation, std_deviation])

    def checkClonesInPop(self, ind, new_pop):
        is_clone = False
        for other_ind in new_pop:
            if (
                ind == other_ind
                and sum(ind) == sum(other_ind)
                and ind.index != other_ind.index
            ):
                is_clone = True
                break
        return is_clone



    def criterio1(self, new_pop, porcentagem, k=30):
        """Seleciona os candidatos ao conjunto elite com base nas diferenças percentuais de aptidão."""

        if self.DEBUG:
            self.cout(f"CRITÉRIO 1 RCE - Selecionando candidatos ao conjunto elite")

        elite_individuals = []

        # Ordenar a população em ordem crescente de aptidão e selecionar os k primeiros indivíduos
        sorted_population = sorted(self.POPULATION, key=lambda x: x.fitness.values[0])

        # Obter o melhor indivíduo (HOF) da população
        best_ind = new_pop[0]
        best_fitness = best_ind.fitness.values[0]
        max_difference = (1 + porcentagem) * best_fitness

        # Selecionar com as menores diferenças percentuais
        for ind in sorted_population:
            if ind.fitness.values[0] <= max_difference:
                elite_individuals.append(ind)
            else:
                break  # Parar a seleção quando a diferença percentual for maior que o limite

        # Colocando na pop aleatória
        if self.DEBUG:
            print(
                f"Calculando percentual de {porcentagem*100}% com base no melhor fitness = {best_fitness} e pegando os {len(elite_individuals)} melhores.\n Porcentagem de {best_fitness} = {max_difference} "
            )

        for i, ind in enumerate(elite_individuals):
            new_pop[i] = self.setup.toolbox.clone(ind)
            new_pop[i].rce = "SIM_1"

        return elite_individuals

    def criterio2_alternative(self, ind_selecionados, delta=6):
        """Comparar as variáveis de decisão de cada indivíduo e verificar se existem diferenças superiores a 'delta'."""
        if self.DEBUG:
            self.cout(
                f"CRITÉRIO 2 - Comparar as variáveis de decisão de cada indivíduo e verificar valores superiores a 'delta' = {delta}."
            )

        self.CONJUNTO_ELITE_RCE.clear()

        for i in range(len(ind_selecionados)):

            #    todo   Calcular Diff - diferença entre as variáveis de indivíduo e individuo lista
            diff = np.array(ind_selecionados[i]) - np.array(ind_selecionados[0])

            # todo retornar true ou false caso ind seja diferente para colocar no array correto
            if sum(diff) > delta:  # ind diferente
                if (ind_selecionados[i] not in self.pop_RCE) and (
                    ind_selecionados[i] not in ind_selecionados[0]
                ):
                    self.pop_RCE.append(self.POP_OPTIMIZATION[i])
                    self.CONJUNTO_ELITE_RCE.add(tuple(self.POP_OPTIMIZATION[i]))
                    ##print("Delta= ", sum(diff))

        #! REVER AQUI SE é APENAS DEBUG MESMO
        if self.DEBUG:
            if not self.pop_RCE:
                print("Nenhum indivíduo atende aos critérios. :( ")


            print("Tamanho Elite = ", len(self.pop_RCE))
            #print("Tamanho Elite = ", len(self.CONJUNTO_ELITE_RCE))

        return self.pop_RCE

    def newCriterio(self, population):
        self.CONJUNTO_ELITE_RCE.clear()
        self.pop_RCE = []

        def criterio1_reduzido(population):
            #! critério 1 e obtém os N melhores com 30% do valor do melhor fitness

            best_ind = self.elitismoSimples(population)[0]
            best_fitness = best_ind.fitness.values[0]
            max_difference = (1 + self.setup.porcentagem) * best_fitness
            if self.DEBUG:
                print(
                    f"Fitness ({self.setup.porcentagem * 100})% = {round(max_difference,3)}"
                )
            return max_difference, best_ind

        if self.DEBUG:
            self.cout(
                "Criterio 1 - Pegando o valor máximo de Fitness para selecionar individuos"
            )
        max_difference, best_ind = criterio1_reduzido(population)
        self.pop_RCE.append(best_ind)

        def calculaDiff(ind, lista):
            count = 0

            # calcula a diferença entre o ind selecionado e o pessoal do RCE
            for i in range(0, len(lista)):
                diff = abs(np.array(ind) - np.array(lista[i]))
                array = list(diff)

                # pegando quantos valores nao sao nulos  -> caso tenha valor limite
                if sum(array) > 0.0:
                    for value in array:
                        if value > self.setup.NUM_VAR_DIF:
                            count += 1

                    # se o contador for maior que delta
                    if count >= self.setup.delta:
                        # print("\nArray diferente")

                        # print(" diff", array)

                        # print("Var decision diferentes = ", count)
                        return True

                    else:
                        return False  # sem diversidade suficiente
                else:
                    return False  # clone: Variaveis iguais

        if self.DEBUG:
            self.cout(f"New - CRITÉRIO 2 RCE ")

        for ind in population:
            # criterio 1
            if ind.fitness.values[0] <= max_difference:
                # criterio 2
                diferente = calculaDiff(ind, self.pop_RCE)  # delta como valor limite
                if diferente:
                    if ind not in self.pop_RCE:
                        self.pop_RCE.append(ind)
                        self.CONJUNTO_ELITE_RCE.add(tuple(ind))


        if self.DEBUG:
            if len(self.pop_RCE) == 1:
                print("Nenhum indivíduo atende aos critérios. :( ")

            print("\nTamanho Elite = ", len(self.pop_RCE))
            #print("Tamanho Elite = ", len(self.CONJUNTO_ELITE_RCE))

        return self.pop_RCE

    def aplicar_RCE(self, generation, current_population):

        #! a - Cria uma pop aleatória (eliminando a pop aleatória criada na execução anterior do RCE)
        new_pop = self.setup.toolbox.population(
            n=self.setup.POP_SIZE
        )  # retorna uma pop com lista de individuos de var de decisão

        # Avaliar o fitness da população atual
        #self.setup.avaliarFitnessIndividuos(current_population)
        self.calculateFitnessGeneration(current_population)

        #! b - Coloca o elite hof da pop anterior  no topo (0)
        pop = self.elitismoSimples(current_population)

        if self.DEBUG:
            print(
                f"Elitismo HOF Index[{pop[0].index}] {pop[0]} \n Fitness = {pop[0].fitness.values} | Diversidade = {sum(pop[0])}"
            )

        new_pop[0] = self.setup.toolbox.clone(pop[0]) # pop[0] é o melhor individuo HOF

        #! Critério 2 usando este array e vai colocando os indivíduos selecionados pelo critério 2 na pop aleatória (passo a)
        ind_diferentes_var = self.newCriterio(
            current_population,
        )

        # cOLOCANDO ATRIBUTOS
        for i, ind in enumerate(ind_diferentes_var, start=0):
            new_pop[0].rce = "HOF"
            if i > 0:
                new_pop[i] = self.setup.toolbox.clone(ind)
                new_pop[i].rce = "SIM"

        #! Criterio 3 retorna pop aleatória modificada (com hof + rce + Aleatorio)
        self.calculateFitnessGeneration(new_pop)
        conjunto_elite = self.generateInfoIndividual(new_pop, generation)

        #! debug
        if self.DEBUG:
            self.cout(f"CRITERIO 3 - População aleatória modificada [HOF,RCE,Aleatorio] ")


        return new_pop

    def elitismoSimples(self, pop):
        self.hof.update(pop)
        pop[0] = self.setup.toolbox.clone(self.hof[0])
        return pop

    def criterio2(self, elite_individuals, delta):
        """Comparar as variáveis de decisão de cada indivíduo e verificar se existem diferenças superiores a 'delta'."""
        self.cout(
            f"CRITÉRIO 2 - Comparar as variáveis de decisão de cada indivíduo e verificar se existem diferenças superiores a 'delta' = {delta}."
        )
        self.pop_RCE = []
        self.CONJUNTO_ELITE_RCE.clear()

        for i in range(len(elite_individuals)):
            current_individual = elite_individuals[i]
            is_diferente = False

            for j in range(i + 1, len(elite_individuals)):
                other_individual = elite_individuals[j]
                diff_counter = 0

                for var_index in range(len(current_individual)):
                    current_var = current_individual[var_index]
                    other_var = other_individual[var_index]

                    if abs(current_var - other_var) > delta:
                        # print(abs(current_var - other_var))
                        diff_counter += 1

                if diff_counter >= 1:
                    is_diferente = True

            if is_diferente:
                # print(f"Indivíduo do tipo {type(current_individual)} VAR({current_individual})\n diferente! adicionado à nova população.")
                self.pop_RCE.append(current_individual)
                self.CONJUNTO_ELITE_RCE.add(tuple(current_individual))



        if self.DEBUG:
            if not self.pop_RCE:
                print("Nenhum indivíduo atende aos critérios. :( ")

            print("Tamanho Elite = ", len(self.pop_RCE))


        return self.pop_RCE

    def calculateFitnessGeneration(self, new_pop):
        # Calculando o fitness para geração
        for ind in new_pop:
            if not ind.fitness.valid:
                fitness_value = self.setup.toolbox.evaluate(ind,self.setup)
                ind.fitness.values = (fitness_value,)

    # def _avaliarFitnessIndividuos(self, pop):
    #     """Avaliar o fitness dos indivíduos da população atual."""
    #     fitnesses = map(self.setup.toolbox.evaluate, pop)
    #     for ind, fit in zip(pop, fitnesses):
    #         if ind.fitness.values:
    #             ind.fitness.values = [fit]





    #! Main LOOP
    def run(self,  RCE=False, num_pop=0):

        population = [self.POPULATION]

        #! Loop principal através das gerações
        for current_generation in range(self.setup.NGEN):

            # Selecionar os indivíduos para reprodução
            offspring = self.setup.toolbox.select(
                population[num_pop], k=len(population[num_pop])
            )

            # Clone the selected individuals
            offspring = [self.setup.toolbox.clone(ind) for ind in offspring]

            # Aplicar crossover
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < self.setup.CXPB:
                    self.setup.toolbox.mate(child1, child2)
                    del child1.fitness.values
                    del child2.fitness.values

            # Aplicar mutação
            for mutant in offspring:
                if random.random() < self.setup.MUTPB:
                    self.setup.toolbox.mutate(mutant)
                    del mutant.fitness.values

            #  Avaliar o fitness dos novos indivíduos
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]

            #! Evaluate each individual separately
            for ind in invalid_ind:
                # call the 'funcao_objetivo_IEEE14' using the current individual attributes
                fitness = self.setup.toolbox.evaluate(ind, self.setup)

                # Assign the fitness value to the individual
                ind.fitness.values = (fitness,)

            # # faz um map dos valores de fitness de cada individuo
            # fitnesses = map(self.setup.toolbox.evaluate, invalid_ind)
            # for ind, fit in zip(invalid_ind, list(fitnesses)):
            #     ind.fitness.values = [fit]


            if self.DEBUG:
                print("ALGORITIMO EVOLUTIVO COM AG PELO DEAP EFEATUADO COM SUCESSO!")

            #! Aplicar RCE
            if RCE and ((current_generation + 1) % self.setup.num_repopulation == 0):
                if self.DEBUG:
                    self.cout(
                        f"RCE being applied! - Generation = {current_generation + 1} ",
                    )
                #!copia pop aleatória modificada retornada para pop atual
                new_population = self.aplicar_RCE(
                    current_generation + 1, offspring
                )

                # Retorna minha nova população com RCE
                population[num_pop][:] = new_population

                # Gera o Excel com a pop com RCE em Excel
                conjunto_elite = self.generateInfoIndividual(population[num_pop][:], current_generation + 1)
                self.show_ind_df(conjunto_elite, "Individuos da nova população aleatória com RCE (Conjunto Elite)")

            else:
                population[num_pop][:] = offspring

            # Registrar estatísticas no logbook
            self.elitismoSimples(population[num_pop])
            self.registrarDados(current_generation)
            record = self.stats.compile(population[num_pop])
            self.logbook.record(gen=current_generation, **record)

        # Retornar população final, logbook e elite
        return population[num_pop], self.logbook, self.hof[0]

    def visualizarPopAtual(self, geracaoAtual, stats):

        for i in range(len(self.POPULATION)):
            datasetIndividuals = {
                "Generations": geracaoAtual + 1,
                "index": i,
                "Variaveis de Decisão": self.POPULATION[i],
                "Fitness": self.POPULATION[i].fitness.values,
                "Media": stats[0],
                "Desvio Padrao": stats[1],
                "RCE": " - ",
            }
            self.allIndividualValuesArray.append(datasetIndividuals)

    def cout(self, msg):
        print(
            "\n=========================================================================================================="
        )
        print("\t", msg)
        print(
            "==========================================================================================================\n"
        )
    def generateInfoIndividual(self, new_pop, generation):
        ind_array = []

        for i, ind in enumerate(new_pop):
            # print(f"Index[{ind.index}] - ind_variables {ind} \n Fitness = {ind.fitness.values} ")

            ind.index = i

            ind_info = {
                "Generations": generation,
                "index": ind.index,
                "Variaveis de Decisão": ind,
                "Fitness": ind.fitness.values[0],
                "RCE": ind.rce,
                "Diversidade": np.sum(ind),
            }

            # Adicionar a informação de clone ao dicionário
            # is_clone = self.checkClonesInPop(ind, new_pop)
            # ind_info["CLONE"] = "SIM" if is_clone else "NAO"

            ind_array.append(ind_info)

        return ind_array

    def show_ind_df(self, array, text, save = True):
        df = pd.DataFrame(array)
        if save:
            df.to_excel(f"pop_final.xlsx")

        if self.DEBUG:
            print(text)
            print(df.head(10))

        # contar quantos SIM na coluna CLONE se a coluna RCE for SIM
        # display(df[df["RCE"] != ""].value_counts())

## 4.1) Classe RedeEletricaPandapower


In [45]:
import pandapower as pp
import pandapower.networks as pw
import pandas as pd
import numpy as np

from rich.console import Console
from rich.theme import Theme
from rich.traceback import install

install()

class Logger:
    def __init__(self):
        self.console = Console(theme=Theme({
            "success": "bold green",
            "warning": "yellow",
            "error": "bold red",
            "info": "white"  # Added "info" level for default blue color
        }))

    def log(self, message, level="info"):  # Changed default level to "info"
        """Logs a message with the specified level and color."""
        if level == "success":
            self.console.print(f"[success]{message}[/]")
        elif level == "warning":
            self.console.print(f"[warning]{message}[/]")
        elif level == "error":
            self.console.print(f"[error]{message}[/]")
        else:
            self.console.print(f"[info]{message}[/]") # Changed to "info" to use blue color



class RedeEletricaPandaPower:
    def __init__(self, network_name, debug=False):
        self.net = self.carregar_redes_padrao(network_name)
        self.debug = debug
        self.console = Logger()

        #metoodos
        self.criar_mapeamento_ramos()

        # global
        self.pesos = {
            "tensao": {"min": 100, "max": 100},
            "loading_linhas": 100,
            "loading_trafos": 150,
            "demanda": 99,
        }
        self.agendamento = pd.DataFrame()
        self.contingencia= pd.DataFrame()

    def carregar_redes_padrao(self,network_name = "14"):
        #!todo -> Switch para as redes disponiveis na lib
        match network_name:
            case "14":
                network = pw.case14()
            case "30":
                #RZ não confundir com case30
                network = pw.case_ieee30()
            case "57":
                # This function provides the ieee case57 network with the data origin PYPOWER
                network = pw.case57()

            case "118":
                network = pw.case118()

            case _:
                print("Rede não encontrada, forneça o numero como string")
                network = None

        return network

    def criar_mapeamento_ramos(self):
        """Mapeia pares de barramentos para índices de linhas e trafos"""
        self.mapeamento_ramos = {
            'linhas': {},
            'trafos': {}
        }

        # Linhas
        for idx, row in self.net.line.iterrows():
            key = tuple(sorted((row['from_bus'], row['to_bus'])))
            self.mapeamento_ramos['linhas'][key] = idx

        # Transformadores
        for idx, row in self.net.trafo.iterrows():
            key = tuple(sorted((row['hv_bus'], row['lv_bus'])))
            self.mapeamento_ramos['trafos'][key] = idx

        return self.mapeamento_ramos



    def validar_dados(self, df_agendamento, df_contingencia):
        """Valida consistência dos dados antes de processar"""
        # Verifica colunas obrigatórias
        required_agendamento = ["ramo", "inicio", "duracao", "prioridade"]
        if not all(col in df_agendamento.columns for col in required_agendamento):
            #raise ValueError("Colunas faltantes no agendamento_df")
            print("Colunas faltantes no agendamento_df")


        # Verifica existência dos ramos
        for _, row in df_agendamento.iterrows():
            ramo = tuple(sorted(row['ramo']))
            if not (ramo in self.mapeamento_ramos['linhas'] or ramo in self.mapeamento_ramos['trafos']):
                #raise ValueError(f"Ramo {row['ramo']} não existe na rede")
                print(f"Ramo {row['ramo']} não existe na rede")

        self.agendamento = df_agendamento
        self.contingencia = df_contingencia


    def hashtableindex (self, carregamento, n_carregamentos, contingencia, n_contingencias, desligamentos):
        """"
        Recebe os dados do cenário e retorna o índice da tabela hash correspondente
        carregamento -> inteiro de 1 a numero de carregamentos
        n_carregamentos -> inteiro com o número total de carregamentos
        contingencia -> inteiro de 1 a numero de contingencias
        n_contingencias -> total de contingencias
        desligamentos -> vetor linha com ndeslig elementos booleanos
        """
        num_desligamentos= len(desligamentos)

        #converte o vetor binário em inteiro de forma eficiente
        #https://stackoverflow.com/questions/24560596/fastest-way-to-convert-a-binary-listor-array-into-an-integer-in-python
        digits = ['0', '1']


        k = int("".join([ digits[y] for y in desligamentos ]), 2)

        return (k * (n_carregamentos) * (n_contingencias) ) + ((carregamento-1) * (n_contingencias))  + (contingencia-1)


    #==============================================================================================================================================================

    #! UTILS
    def log(self, mensagem,level="info"):
        if self.debug:
            self.console.log(mensagem,level)

    def plot_rede_eletrica_status(self, x, y, net, trafo_status, line_status):
        """Plota o status (ligado/desligado) dos transformadores e linhas em um gráfico de barras.

        Args:
            x: Dados para o eixo x.
            y: Dados para o eixo y.
            net: Objeto pandapower contendo os resultados da simulação.
            trafo_status: Array contendo o status (True/False) dos transformadores.
            line_status: Array contendo o status (True/False) dos linhas.
        """

        # Combinar os status de transformadores e linhas
        all_status = np.concatenate([trafo_status, line_status])

        # Criar um array de rótulos para o eixo x (nomes dos ramos)
        num_trafos = len(trafo_status)
        num_lines = len(line_status)
        x_labels = np.arange(1, num_trafos + num_lines + 1)

        # Criar o gráfico de barras
        fig, ax = plt.subplots(figsize=(10, 6))  # Ajustar o tamanho conforme necessário

        # Definir cores para ligado (verde) e desligado (vermelho)
        colors = ['g' if s else 'r' for s in all_status]

        # Criar as barras com as cores definidas
        bars = ax.bar(x_labels, np.ones_like(all_status), color=colors)

        # Configurar o gráfico
        ax.set_xlabel("Ramo")
        ax.set_ylabel("Status")
        ax.set_title("Status dos Transformadores e Linhas (Ligado/Desligado)")
        ax.set_xticks(x_labels)  # Definir os ticks do eixo x
        ax.set_yticks([1])  # Mostrar apenas o tick '1' no eixo y
        ax.set_yticklabels([''])  # Remover o rótulo '1' do eixo y

        # Ajustar a legenda para mostrar 'Transformador' e 'Linha'
        legend_labels = ['Ligado', 'Desligado']
        legend_handles = [bars[0], bars[-1] if not all_status[0] else bars[all_status.index(False)]]

        ax.legend(legend_handles, legend_labels, loc='best')  # Adicionar legenda

        plt.show()  # Mostrar o gráfico

    def plot_trafo_status_only(self,net):
        """Plota o status (ligado/desligado) dos transformadores e linhas em um gráfico de barras.

        Args:
            net: Objeto pandapower contendo os resultados da simulação.
        """

        # Extrair o status (in_service) dos transformadores e linhas
        trafo_status = net.trafo["in_service"].values
        line_status = net.line["in_service"].values

        # Combinar os status de transformadores e linhas
        all_status = np.concatenate([trafo_status, line_status])

        # Criar um array de rótulos para o eixo x (nomes dos ramos)
        num_trafos = len(trafo_status)
        num_lines = len(line_status)
        x_labels = np.arange(1, num_trafos + num_lines + 1)

        # Criar o gráfico de barras
        fig, ax = plt.subplots(figsize=(10, 6))  # Ajustar o tamanho conforme necessário

        # Definir cores para ligado (verde) e desligado (vermelho)
        colors = ['g' if s else 'r' for s in all_status]

        # Criar as barras com as cores definidas
        bars = ax.bar(x_labels, np.ones_like(all_status), color=colors)

        # Configurar o gráfico
        ax.set_xlabel("Ramo")
        ax.set_ylabel("Status")
        ax.set_title("Status de todas as Linhas (Ligado/Desligado)")
        ax.set_xticks(x_labels)  # Definir os ticks do eixo x
        ax.set_yticks([1])  # Mostrar apenas o tick '1' no eixo y
        ax.set_yticklabels([''])  # Remover o rótulo '1' do eixo y

        # Ajustar a legenda para mostrar 'Transformador' e 'Linha'
        legend_labels = ['Ligado', 'Desligado']
        legend_handles = [bars[0], bars[-1] if not all_status[0] else bars[all_status.index(False)]]

        ax.legend(legend_handles, legend_labels, loc='best')  # Adicionar legenda

        plt.show()  # Mostrar o gráfico

    def plot_trafo_status(self, net):
        """Plota o status (ligado/desligado) dos transformadores em um gráfico de barras.

        Args:
            net: Objeto pandapower contendo os resultados da simulação.
        """

        # Extrair os dados de loading_percent e status (in_service)
        loading_percent = net.res_trafo["loading_percent"].values
        status = net.trafo["in_service"].values

        # Criar um array de rótulos para o eixo x (nomes dos ramos)
        x_labels = np.arange(1, len(loading_percent) + 1)

        # Criar o gráfico de barras
        fig, ax = plt.subplots(figsize=(10, 6))  # Ajustar o tamanho conforme necessário
        bars = ax.bar(x_labels, loading_percent, color='b')  # Barras azuis para loading_percent

        # Ajustar as cores das barras para vermelho se o transformador estiver desligado
        for i, bar in enumerate(bars):
            if not status[i]:  # Se status for False (desligado)
                bar.set_color('r')

        # Configurar o gráfico
        ax.set_xlabel("Ramo")
        ax.set_ylabel("Loading Percent")
        ax.set_title("Status dos Transformadores")
        ax.set_xticks(x_labels)  # Definir os ticks do eixo x
        ax.legend([bars[0], bars[-1]], ['Ligado', 'Desligado'], loc='best')  # Adicionar legenda

        plt.show()  # Mostrar o gráfico


    def show_status(self):

        if self.debug:
            print("="*80)
            print("Rede atual")
            print("="*80)

            print("\nStatus Linhas")
            display(self.net.line[["from_bus","to_bus","in_service"]])
            #display(self.net.res_line[["loading_percent"]])

            # Plot do loading percente das linhas
            #self.plot_trafo_status_only(self.net)
            try:

                # Pega os dados estado e lgiados de linhas e trafos
                trafo_status = self.net.trafo["in_service"].values
                line_status = self.net.line["in_service"].values


                # plot que mostra as linhas ligadas
                self.plot_rede_eletrica_status(x, y, self.net, trafo_status, line_status)


                print("\nStatus Transformadores")
                display(self.net.trafo[["hv_bus","lv_bus","in_service"]])
                #display(self.net.res_trafo[["loading_percent"]])

                self.plot_trafo_status(self.net)
            except Exception as erro:
                print("Erro ao plotar", erro)


            ## Barramentos
            #print("\nTensões nos Barramentos (pu):")
            #display(self.net.res_bus[['vm_pu']])


            ## linhas
            #print("\nPorcentagem de Carga nas Linhas (%):")
            #display(self.net.res_line[['loading_percent']])



            #! PLOT DOS VALORES DE POTENCIA APARENTE POS FLUXO DE POTENCIA
            #self.plot_power_data()

            #print("\nPotência Aparente nas Linhas (MVA):")
            #display(self.net.res_line[['p_from_mw', 'q_from_mvar']])


            # transformadores
            #print("\nPotencia aparente nos transformadores")
            #display(self.net.res_trafo[['p_hv_mw', 'q_hv_mvar', 's_aparente_hv_mva', 'p_lv_mw', 'q_lv_mvar', 's_aparente_lv_mva']])


            #print("\nPorcentagem de Carga nos transformadores (%):")
            #display(self.net.res_trafo[['loading_percent']])

            #print("="*80)


    def plot_power_data(self, index = None):
        """Plota os dados de potência dos barramentos, linhas e transformadores em um gráfico de linhas.

        Args:
            index: Um array contendo os valores do índice da série temporal (tempo ou número da iteração).
        """

        index = np.arange(len(self.net.res_line['p_from_mw']))


        # Extrair os dados de cada elemento da rede com nomes mais descritivos
        line_p_from_mw_data = self.net.res_line['p_from_mw'].values

        index = line_p_from_mw_data


        line_q_from_mvar_data = self.net.res_line['q_from_mvar'].values
        trafo_p_hv_mw_data = self.net.res_trafo['p_hv_mw'].values
        trafo_q_hv_mvar_data = self.net.res_trafo['q_hv_mvar'].values
        #trafo_s_aparente_hv_mva_data = self.net.res_trafo['s_aparente_hv_mva'].values
        #time_series = self.net['time'].values  # Or self.net['iteration'].values
        #print(time_series)

        # Criar o gráfico de linhas
        fig, ax = plt.subplots(figsize=(10, 6))

        # Plotar cada grandeza com uma cor diferente e rótulos descritivos
        ax.plot(index, line_p_from_mw_data, label='Potência Ativa Linhas (MW)', color='blue')
        ax.plot(index, line_q_from_mvar_data, label='Potência Reativa Linhas (MVar)', color='red')
        ax.plot(index, trafo_p_hv_mw_data, label='Potência Ativa Trafos (MW)', color='green')
        ax.plot(index, trafo_q_hv_mvar_data, label='Potência Reativa Trafos (MVar)', color='orange')
        #ax.plot(index, trafo_s_aparente_hv_mva_data, label='Potência Aparente Trafos (MVA)', color='purple')


        # Configurar o gráfico
        ax.set_xlabel("Tempo/Iteração")
        ax.set_ylabel("Valores")
        ax.set_title("Dados de Potência")
        ax.legend()
        plt.grid(True)
        plt.show()



    #! Otimização
    def calcular_violacoes_fitness(self):
        """
        Calcula as violações nos barramentos, linhas e transformadores.

        Considerando um peso para cada grandeza : dois pesos para tensão (max e min) e outro para loading_percent das linhas.

        Somar (valor - limite max ou limite min - valor) para calcular a aptidão daquele cenário.

        Retornar o somatório de todas as violações, mas ao soma cada violação você deve multiplicar por um peso para determinar a aptidão do cenário.

        """
        violacoes = {
            "tensao_barramentos_min": 0,
            "tensao_barramentos_max": 0,
            "loading_linhas": 0,
            "loading_trafos": 0,
        }

        # Verificar tensões nos barramentos (pu) - check
        for idx, row in self.net.res_bus.iterrows():
            # Certifique-se de que a tensão está em pu
            tensao_pu = row["vm_pu"]

            limite_max = self.net.bus.at[idx, "max_vm_pu"]
            limite_min = self.net.bus.at[idx, "min_vm_pu"]

            if tensao_pu > limite_max:
                violacoes["tensao_barramentos_max"] += tensao_pu - limite_max
            elif tensao_pu < limite_min:
                violacoes["tensao_barramentos_min"] += limite_min - tensao_pu

        # Verificar carregamento das linhas
        for idx, row in self.net.res_line.iterrows():

            carregamento = row["loading_percent"] #* 100
            limite_max = self.net.line.at[idx, "max_loading_percent"]

            if carregamento > limite_max:
                self.log("\n\nUltrapassou limite maximo nas linhas",level = "warning")
                self.log(f"{carregamento:.2f} > {limite_max} %",level = "warning")
                #RZ - as violações também devem considerar 100% = 1, deve-se dividir
                violacoes["loading_linhas"] += (carregamento - limite_max) #/ 100

        # Verificar carregamento dos transformadores
        for idx, row in self.net.res_trafo.iterrows():
            carregamento = row["loading_percent"] #* 100
            limite_max = self.net.trafo.at[idx, "max_loading_percent"]

            if carregamento > limite_max:
                self.log("\n\nUltrapassou limite maximo nos transformadores",level = "warning")
                self.log(f"{carregamento:.2f} > {limite_max} %",level = "warning")
                #RZ - as violações também devem considerar 100% = 1, deve-se dividir
                violacoes["loading_trafos"] += (carregamento - limite_max) #/ 100

        #! TODO -> PASSAR OS PESOS NA INSTANCIA DO OBJETO COM VALOR DEFAULT
        """
        Pesos das violações : (Pdem=99,Pv = 100, Pn = 100 e Pe = 150)
        onde PV é a violação de tensão max e min,
        Pdem é para não convergência do fluxo
        Pn e Pe são para o fluxo de potência (pode usar só Pn que depois explico o que é Pe).
        """


        fitness = (
            self.pesos["tensao"]["min"] * violacoes["tensao_barramentos_min"]
            + self.pesos["tensao"]["max"] * violacoes["tensao_barramentos_max"]
            + self.pesos["loading_linhas"] * violacoes["loading_linhas"]
            + self.pesos["loading_trafos"] * violacoes["loading_trafos"]
        )

        #pega as violacoes e transforma em um DF
        violacoes_df = pd.DataFrame([violacoes])

        if self.debug:
            print("\nTotal de violações e salvando num banco de dados...")
            display(violacoes_df)
            self.console.log(f"\n\nAptidão do cenário nos barramentos, linhas e transformadores ", level = "success")
            self.console.log(f"VIOLAÇÃO TOTAL  = {fitness:.2f}\n", level = "success")

        return fitness, violacoes_df


    def calcular_perfil(self,j, ls, le, ms, me, hs, he):
        """
        Calcula o perfil de carregamento (leve, médio ou pesado) para a hora `j`.

        Args:
            j (int): Hora atual.
            ls, le, ms, me, hs, he (int): Limites de horários para os perfis de carga.

        Returns:
            int: Perfil de carregamento (1 = leve, 2 = padrão, 3 = pesado).
            perfil de carga media é igual IEEE_14
            perfil de carga pesada = multiplicar todas as potencias ativas e reativas, identificando os elementos das estruturas de self.net da classe RedeEletrica do pandapower

        """
        if ls <= j % 24 < le:
            return 1  # Leve
        elif ms <= j % 24 < me:
            return 2  # Médio
        elif hs <= j % 24 < he:
            return 3  # Pesado
        return 0  # Fora dos horários definidos


    def avalia_cenarios(self, horas: int, hora_inicio: list, duracao: list, ls, le, ms, me, hs, he , debug = False):
        """
        Args:
            horas (int): Horas de duração da janela de tempo.
            hora_inicio (list): Vetor de horários iniciais dos desligamentos (valores de 0 a m-1).
            duracao (list): Vetor de duração em horas de cada desligamento.
            ls, le, ms, me, hs, he (int): Limites iniciais e finais dos horários de carregamento leve, médio e pesado.

        Returns:
            list: Matriz que armazena todos os cenários do agendamento.
        """

        matriz_cenarios = []
        num_desligamentos = len(hora_inicio)

        # Ajusta limite da janela de tempo se algum desligamento terminar fora da janela
        for i in range(num_desligamentos):
            if horas < (hora_inicio[i] + duracao[i]):
                horas = hora_inicio[i] + duracao[i]

        # Inicializa matrizes auxiliares
        matriz_desligamentos_horas = np.zeros((num_desligamentos, horas), dtype=int)
        matriz_horas = np.zeros(horas, dtype=int)

        # =================== Avaliando desligamentos por hora =================
        for j in range(horas):  # Para cada hora
            for k in range(num_desligamentos):  # Para cada desligamento
                if hora_inicio[k] <= j < (hora_inicio[k] + duracao[k]):
                    matriz_desligamentos_horas[k, j] = 1
                matriz_horas[j] += matriz_desligamentos_horas[k, j] * (2 ** k)

        # =================== Avaliando cenários =================
        for horario in range(horas):
            if horario == 0:  # Condição inicial
                if matriz_horas[horario] > 0:
                    # Se há pelo menos um desligamento ativo
                    perfil = self.calcular_perfil(horario, ls, le, ms, me, hs, he)
                    matriz_cenarios.append([perfil] + matriz_desligamentos_horas[:, horario].tolist())
            else:
                if matriz_horas[horario] != matriz_horas[horario - 1] and matriz_horas[horario] > 0:
                    # Nova topologia
                    perfil = self.calcular_perfil(horario, ls, le, ms, me, hs, he)
                    matriz_cenarios.append([perfil] + matriz_desligamentos_horas[:, horario].tolist())
                else:
                    # Mesmo cenário, mas perfil pode mudar
                    if horario - 1 in [ms, hs] and matriz_cenarios:  # Verifica se matriz_cenarios não está vazia
                        perfil = self.calcular_perfil(horario, ls, le, ms, me, hs, he)
                        matriz_cenarios[-1][0] = max(matriz_cenarios[-1][0], perfil)

        if self.debug:
            self.log("\nMatriz Cenarios:")
            for linha in matriz_cenarios:
                self.log(linha)
            self.log(f"Avaliando um total de {len(matriz_cenarios)} cenários ")


        return matriz_cenarios

    #! Pandapower New metodos
    def executar_fluxo_de_carga(self):
        """
        Executa o fluxo de carga na rede elétrica usando o algoritmo Newton-Raphson.

        Retorna:
            bool: True se o fluxo de carga convergiu, False caso contrário.
        """
        try:
            pp.runpp(self.net, algorithm="nr")
            self.log("\nFluxo de potência executado com sucesso.",level = "success")

            return True
        except pp.LoadflowNotConverged:
            self.console.log("\nErro: Fluxo de potência não convergiu.", level = "error")
            Pdem = 99

            self.calcular_violacoes_fitness()


            return False

    def ajustar_cargas(self, perfil):
        """Ajusta as cargas conforme o perfil (1 = leve, 2 = médio, 3 = pesado)."""
        tipo = ""

        if perfil == 1:
            fator = 0.941  # Carga leve

            tipo = "leve"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        elif perfil == 2:
            fator = 1.0  # Carga média (IEEE14)

            tipo = "media"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        elif perfil == 3:
            fator = 1.177  # Carga pesada

            tipo = "pesada"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        else:
            fator = 1.0  # Perfil padrão (IEEE14)

            tipo = "padrão"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        # usando o scaling
        #self.net.load["p_mw"] *= fator
        #self.net.load["q_mvar"] *= fator
        self.net.load.scaling = fator
        #self.net.gen['vm_pu'] = 1.045
        self.net.gen.scaling = fator

        self.log("Cargas ajustadas.", level = "success")

    def desligar_elementos_agendamento(self, estados):
        """Desliga os elementos (linhas e trafos) com base no cenário."""

        linhas_desligar = []
        trafos_desligar = []

        for i, estado in enumerate(estados):
            if estado == 1:  # Verifica se o ramo deve ser desligado

                ramo = self.agendamento.iloc[i]["ramo"]

                #ramo = agendamento_df.iloc[i]["ramo"]  # Obtém o ramo da tabela

                #self.log("Ramo selecionado",ramo)

                for k in range(len(self.net.line)):
                    # TODO Verifica se o ramo é uma linha ou um transformador
                    if (self.net.line["from_bus"][k] == ramo[0] and self.net.line["to_bus"][k] == ramo[1]) or (self.net.line["from_bus"][k] == ramo[1] and self.net.line["to_bus"][k] == ramo[0] ):
                        linhas_desligar.append(ramo)  # Adiciona o ramo à lista de linhas

                for k in range(len(self.net.trafo)):
                    if (self.net.trafo["hv_bus"][k] == ramo[0] and self.net.trafo["lv_bus"][k] == ramo[1]) or (self.net.trafo["hv_bus"][k] == ramo[1] and self.net.trafo["lv_bus"][k] == ramo[0] ):
                        trafos_desligar.append(ramo)  # Adiciona o ramo à lista de trafos

        self.log(f"\n\nLinhas a serem desligadas: {linhas_desligar}")
        self.log(f"Trafos a serem desligados: {trafos_desligar}\n")

        # Desliga as linhas e trafos encontrados
        self.desligar_elementos(linhas_desligar, trafos_desligar)



    def desligar_contingencia(self, ramo):
        linhas_desligar = []
        trafos_desligar = []

        self.log("Ramo selecionado",ramo)

        for k in range(len(self.net.line)):
            # TODO Verifica se o ramo é uma linha ou um transformador
            if (self.net.line["from_bus"][k] == ramo[0] and self.net.line["to_bus"][k] == ramo[1]) or (self.net.line["from_bus"][k] == ramo[1] and self.net.line["to_bus"][k] == ramo[0] ):
                    linhas_desligar.append(ramo)  # Adiciona o ramo à lista de linhas

        for k in range(len(self.net.trafo)):
            if (self.net.trafo["hv_bus"][k] == ramo[0] and self.net.trafo["lv_bus"][k] == ramo[1]) or (self.net.trafo["hv_bus"][k] == ramo[1] and self.net.trafo["lv_bus"][k] == ramo[0] ):
                    trafos_desligar.append(ramo)  # Adiciona o ramo à lista de trafos

        self.log(f"\n\nLinhas a serem desligadas: {linhas_desligar}")
        self.log(f"Trafos a serem desligados: {trafos_desligar}\n")

        # Desliga as linhas e trafos encontrados
        self.desligar_elementos(linhas_desligar, trafos_desligar)

    def desligar_elementos(self, linhas_desligar, trafos_desligar):
        # Itera pelas linhas a serem desligadas e as desliga na rede
        if not linhas_desligar:
            self.log("Nenhuma linha para desligar")

        else:
            for l in linhas_desligar:

                # Encontra o índice da linha com base em from_bus e to_bus
                index_linha = self.net.line.loc[(self.net.line['from_bus'] == l[0]) & (self.net.line['to_bus'] == l[1])].index

                # Verifica se o índice foi encontrado (CORRIGIDO AQUI PVRV)
                if not index_linha.empty:
                    # Desliga a linha usando o índice encontrado
                    self.net.line.loc[index_linha, 'in_service'] = False

                    #print(f"Linha {l} desligada com sucesso.")


                else:
                    print(f"Linha {l} não encontrada na rede.")


        # Itera pelos transformadores a serem desligados e os desliga na rede
        if not trafos_desligar:
            self.log("Nenhum transformador para desligar")
        else:
            for t in trafos_desligar:

                # Encontra o índice da linha com base em from_bus e to_bus
                index_trafo = self.net.trafo.loc[(self.net.trafo['hv_bus'] == t[0]) & (self.net.trafo['lv_bus'] == t[1])].index

                # Verifica se o índice foi encontrado
                if not index_trafo.empty:
                    # Desliga a linha usando o índice encontrado
                    self.net.trafo.loc[index_trafo, 'in_service'] = False

                    #print(f"Transformador {t} desligado com sucesso.")

                else:
                    print(f"Transformador {t} não encontrado na rede.")


    #! Old Pandapower

    #! Funções matematicas
    def calcular_potencia_aparente_trafos(self):
        """Calcula a potência aparente nos transformadores."""
        if not self.net.res_trafo.empty:
            if 's_aparente_hv_mva' not in self.net.res_trafo.columns:
                self.net.res_trafo['s_aparente_hv_mva'] = 0
            if 's_aparente_lv_mva' not in self.net.res_trafo.columns:
                self.net.res_trafo['s_aparente_lv_mva'] = 0

            # Calculando potência aparente para alta e baixa tensão
            potencia_high_tensao = (self.net.res_trafo['p_hv_mw']**2 + self.net.res_trafo['q_hv_mvar']**2)**0.5
            potencia_baixa_tensao = (self.net.res_trafo['p_lv_mw']**2 + self.net.res_trafo['q_lv_mvar']**2)**0.5

            return potencia_high_tensao, potencia_baixa_tensao
        else:
            print("Nenhum transformador na rede para calcular potência aparente.")
            return []

    def calcular_potencia_aparente_linhas(self):
        """Calcula a potência aparente nas linhas."""
        return (self.net.res_line['p_from_mw']**2 + self.net.res_line['q_from_mvar']**2)**0.5


    def religar_todos_os_ramos_agendamento(self):
        """Religa todos os ramos (linhas e transformadores) da rede elétrica."""
        # Religa todas as linhas
        self.net.line['in_service'] = True

        # Religa todos os transformadores
        self.net.trafo['in_service'] = True

        self.log("Todos os ramos religados.", level="success")



    def imprimir_resultados(self):
        """Retorna um array com todos os dados da rede elétrica"""
        dataframe = pd.DataFrame()

        # Calculo de potencia e colocando uma nova tabela no pandapower
        high_power_transformador, lower_power_transformador = self.calcular_potencia_aparente_trafos()
        fluxo_potencia_aparente_linhas = self.calcular_potencia_aparente_linhas()

        self.net.res_trafo['s_aparente_hv_mva'] = high_power_transformador
        self.net.res_trafo['s_aparente_lv_mva'] =  lower_power_transformador

        if self.debug:

            self.show_status()


        dataframe["tensao_nos_barramentos"] =  self.net.res_bus[['vm_pu']]
        dataframe["potencia_aparente_nas_linhas"] =  fluxo_potencia_aparente_linhas
        dataframe["porcentagem_de_carga_nas_linhas"] =  self.net.res_line[['loading_percent']]
        dataframe["potencia_aparente_nos_transformadores"] =  self.net.res_trafo[['p_hv_mw']]
        dataframe["porcentagem_de_carga_nos_transformadores"] =  self.net.res_trafo[['loading_percent']]

        dataframe.to_excel("dados_rede_eletrica.xlsx")
        self.log("\n\n\nDados da rede eletrica em formato de tabela excel disponivel!")

        return dataframe


## 4.2) Simulação e Modelagem Matematica com IEEE 14 e Pandapower

In [46]:





#! TODO -> (26/05/25) Função implementada em Março mas precisa de paralelismo para ficar mais eficiente e melhor uso da hash table
def funcao_objetivo_IEEE14(individuo, setupobj, _debug = False):

    #! 1) Criar a rede elétrica IEEE 14 barras, Inicializar a classe com a rede e carrega a tabela de agendamento
    rede = RedeEletricaPandaPower("14", debug=_debug)


    #! Colocando pesos como input do usuario e os dados de entrada do agendamento
    rede.pesos["tensao"] = {"min": 100, "max": 100}
    rede.pesos["loading_linhas"] = 100
    rede.pesos["loading_trafos"] = 100


    #=====================================================

    # Tabela agendamentos em xlsx hardcoded
    #! TODO -> Colocar dados de agendamento e contigencia fora da funcao objetivo
    #! TODO -> Criação de uma nova função para a hashtable baseado do tipo de Rede fora da função objetivo no Setup
    #! TODO -> Usar consulta na hashtbale na base de dados do Setup dentro da funcao objetivo
    #! TODO -> Configuração de Hash Table e Tabela de agendamentos fora da funcção objetivo mas recebe os parametros do Setup para dentro da funcao objetivo com consulta de hash

    agendamento_df = pd.DataFrame([
        {"ramo": [1, 4], "inicio": "14:00", "duracao": 6 ,"prioridade": 4},
        {"ramo": [1, 3], "inicio": "15:00", "duracao": 5, "prioridade": 1},
        {"ramo": [3, 6], "inicio": "14:00", "duracao": 6, "prioridade": 1},
        {"ramo": [11, 12], "inicio": "18:00", "duracao": 6, "prioridade": 1},
        {"ramo": [9, 10], "inicio": "15:00", "duracao": 4, "prioridade": 1}
    ])

    contingencia_df = pd.DataFrame([
            {"contingencia":1,  "from":2 , "to": 3},
            {"contingencia":2,  "from":5 , "to": 12},
            {"contingencia":3,  "from":12 , "to": 13},
    ])

    # Converter horários de início para horas do dia
    agendamento_df['inicio'] = agendamento_df['inicio'].apply(lambda x: int(x.split(':')[0]))

    # Calcular horário de término em horas do dia
    agendamento_df['final'] = agendamento_df.apply(lambda row: (row['inicio'] + row['duracao']) % 24, axis=1)

    # Calcular a duração total do agendamento em horas
    duracao_total_agendamento = (agendamento_df['inicio']+agendamento_df['duracao']).max()
    rede.validar_dados(agendamento_df, contingencia_df)

    # passando a variavel de decisão na função objetivo
    agendamento_df["inicio"] = individuo

    #! Calculo  de otimização para achar o fitness de cada cenario
    violacoes_total = []
    violacoes_hash_table = {}

    # Generate hash key (teste 01)
    contingencias = contingencia_df['contingencia'].to_list()
    num_carregamentos = 3
    num_contingencias = len(contingencias) # 3
    num_desligamentos = len(agendamento_df) # 5

    # FAZENDO UM BANCO EM MEMORIA DE EXECUÇÃO
    #bd_aptidao_cenario = [-1.0]*(num_contingencias* num_carregamentos*(2**num_desligamentos) )

    #=====================================================



    # 2)  Avaliar cenários e criar matriz de cenários
    matriz_cenarios = rede.avalia_cenarios(
            horas = duracao_total_agendamento,
            hora_inicio=agendamento_df['inicio'],
            duracao=agendamento_df['duracao'],
            ls=0, le=8,
            ms=8, me=18,
            hs=18, he=24
        )

    try:
        # 3) Processar cada cenário da matriz de cenários
        for cenario in matriz_cenarios:
            perfil = cenario[0]
            estado_ramos = cenario[1:]

            # 4) Ajustar carregamento para o perfil do cenário
            rede.ajustar_cargas(perfil)

            # Loop through contingencies before calculating violations for the scenario
            for contingencia_atual in range(num_contingencias):
                contingencia_atual += 1
                hash_key = rede.hashtableindex(perfil, num_carregamentos, contingencia_atual, num_contingencias, estado_ramos)
                #RZ_01jun2025 - verifica se o cenário já foi calculado na tabela hash
                if setupobj.tabela_hash[hash_key] < 0.0:

                  #5)  Ligar todos os ramos antes de aplicar mudanças
                  rede.religar_todos_os_ramos_agendamento()

                  # 6) Fazendo os deligamentos com base na tabela em .xlsx e nos cenários calculados
                  rede.desligar_elementos_agendamento(estado_ramos)

                  # 7) Identifica ramos afetados pela contingência
                  ramo_contingencia = list(contingencia_df.loc[contingencia_df['contingencia'] == contingencia_atual, ['from', 'to']].values[0])
                  rede.log(f"\n{contingencia_atual}) Ramo da contingencia = { ramo_contingencia}\n")

                  # 8) Desliga os ramos afetados
                  rede.desligar_contingencia(ramo_contingencia)

                  # 9) Executar fluxo de potência para o cenário com contingência
                  if rede.executar_fluxo_de_carga():

                      # 10) Calcular violações com pesos e armazenar os resultados
                      fitness, violacoes_df = rede.calcular_violacoes_fitness()

                  else:
                      fitness = rede.pesos["demanda"] # penalidade com valor default de 99

                  setupobj.tabela_hash[hash_key] = fitness

                  #! 11) Store violation in the hash table
                  violacoes_hash_table[hash_key] = fitness
                  rede.log(f"Hash key = { hash_key}\n")

                else:
                  fitness = setupobj.tabela_hash[hash_key]

                violacoes_total.append(fitness)

            #! Ver apenas o true in service de barras e transformadores
            rede.show_status()

        if _debug:
          #! Usando dicionario nos temos os valores acumulando tirando os valores nulos
          hash_df2 = pd.DataFrame(violacoes_hash_table.items(), columns=['Hash Key', 'Fitness'])

          # Passando os valores do array direto no dataframe com os index como chave (hash = chave, valor)
          hash_df = pd.DataFrame(setupobj.tabela_hash, columns=[ 'Fitness'])
          filtered_hash_table = hash_df.loc[hash_df['Fitness'] > 0]

          hash_df.to_excel("hash_table.xlsx", index=False)

        # 12) Calcular fitness final com somatorio das vioações com pesos de todos os cenarios
        fitness_final = sum(violacoes_total)
        rede.log(f"\nFitness do agendamento = {fitness_final:.2f}\n")
        return fitness_final

    except Exception as e:
        print(f"\nErro: {e}")

<p align="center">
  <img src="https://media.licdn.com/dms/image/v2/D4D22AQFMN4mytV8VlQ/feedshare-shrink_1280/B4DZWFGmGyG4Ao-/0/1741694854673?e=1750896000&v=beta&t=8oWhx1CObYXT0MlkDO0udm1iBEX804kwZLi7CSgYTYw" width="400"/>
  <img src="https://electricgrids.engr.tamu.edu/wp-content/uploads/sites/129/2017/04/IEEE-14-Bus-System.png" width="500"/>
</p>

## 5) Teste algoritmo para simulação de rede IEEE 14 com RCE Framework

In [47]:

 
def entrada_de_dados():
    # Tabela agendamentos em xlsx hardcoded
    agendamento_df = pd.DataFrame([
            {"ramo": [1, 4], "inicio": "14:00", "duracao": 6 ,"prioridade": 4},
            {"ramo": [1, 3], "inicio": "15:00", "duracao": 5, "prioridade": 1},
            {"ramo": [3, 6], "inicio": "14:00", "duracao": 6, "prioridade": 1},
            {"ramo": [11, 12], "inicio": "18:00", "duracao": 6, "prioridade": 1},
            {"ramo": [9, 10], "inicio": "15:00", "duracao": 4, "prioridade": 1}
        ])

    contingencia_df = pd.DataFrame([
                {"contingencia":1,  "from":2 , "to": 3},
                {"contingencia":2,  "from":5 , "to": 12},
                {"contingencia":3,  "from":12 , "to": 13},
        ])

        # Generate hash key (teste 01)
    contingencias = contingencia_df['contingencia'].to_list()
    num_carregamentos = 3
    num_contingencias = len(contingencias)  # 3
    num_desligamentos = len(agendamento_df)  # 5
    
    return {
        "contigencias": contingencias,
        "num_carregamentos": num_carregamentos,
        "num_contingencias": num_contingencias,
        "num_desligamentos": num_desligamentos,
        "horarios_agendamento": agendamento_df,
         
    }
    


    #! Entrada de dados simulando que seja uma planillha em excel
dados = entrada_de_dados()



# Instanciando os Objetos
setup = Setup(params, fitness_function = funcao_objetivo_IEEE14,
            
tamanho_hash=(dados["num_contingencias"] * dados["num_carregamentos"]*(2**dados["num_desligamentos"])))




[DEBUG] Dados do problema =  [14, 15, 14, 18, 15] <function funcao_objetivo_IEEE14 at 0x721e899cc9a0>


In [48]:
funcao_objetivo_IEEE14(
    individuo = [14, 15, 16, 17, 18],  # Exemplo de indivíduo com horários de desligamento
    setupobj = setup,
    _debug = True  # Ativar debug para ver os detalhes do processo   
)

Matriz Cenarios:

[2, 1, 0, 0, 0, 0]

[2, 1, 1, 0, 0, 0]

[2, 1, 1, 1, 0, 0]

[2, 1, 1, 1, 1, 0]

...

Rede atual

Status Linhas


,from_bus,to_bus,in_service
0,0,1,True
1,0,4,True
2,1,2,True
3,1,3,True
4,1,4,True
5,2,3,True
6,3,4,True
7,5,10,True
8,5,11,True
9,5,12,True


Erro ao plotar name 'x' is not defined


Fitness do agendamento = 144.64

np.float64(144.64455718657427)

In [49]:




def run_framework():
    """Função principal para executar o framework de otimização."""
    
    
    from datetime import datetime
    start = datetime.now()
    
    #! Entrada de dados simulando que seja uma planillha em excel
    dados = entrada_de_dados()

    # Instanciando os Objetos
    setup = Setup(params, fitness_function = funcao_objetivo_IEEE14,
                  tamanho_hash=(dados["num_contingencias"] * dados["num_carregamentos"]*(2**dados["num_desligamentos"])))

    alg = AlgoritimoEvolutivoRCE(setup, DEBUG = True)


    # Loop Algoritmo Evolutivo podendo receber a função objetivo e as variaveis do problema
    pop_with_repopulation, logbook_with_repopulation, best_variables = alg.run(
        RCE=True,
    )
    
    
    
    end = datetime.now()
    elapsed = end - start
    print(f"Elapsed Time : {elapsed}")
    print("\n\nEvolução concluída  - 100%")
    
    
    
    
    # Consolidado output console e dashboard
    f = open("output.txt", "w")
    print(best_variables, file=f)
    print(elapsed, file=f)
    #print(pop_with_repopulation, file=f)
    print(logbook_with_repopulation, file=f)
    f.close()

    # Passando os valores do array direto no dataframe com os index como chave (hash = chave, valor)
    hash_df1 = pd.DataFrame(setup.tabela_hash, columns=['Fitness'])
    hash_df1.to_excel("hash_table.xlsx", index=False)

    # # Resultados
    x, y, z, fig = alg.dashboard.visualize(
         logbook_with_repopulation, pop_with_repopulation,
     )



#run_framework()